# <span style="color:#2E86C1">Prova final i previsió de juliol 2026</span>

Codi de la previsió de juliol, la recomanació d'estoc i la de campanya. Les conclusions estan a la memòria del projecte (Bloc 4).

### <span style="color:#D68910">5.1 Previsió de juliol 2026</span>

Apliquem el **model congelat** (Ridge, `alpha = 300`, entrenat amb 2022–2024) als 18 casos de `sales_forecast_input.csv`.

- **Clau completa:** `month` + `channel` + `product_line` + `region` (una fila per combinació).
- **`prediccio_unitats`:** el que diu el model.
- **`venda_real`:** buida; s'omplirà quan acabe juliol per a comparar.
- **`tipus`:** `previsió`, per a no confondre-la amb vendes observades (també en Power BI).

Resultat guardat a `evidencies/prediccions_juliol_2026.csv`.

In [1]:
import pandas as pd
import numpy as np

KEY = ['channel', 'product_line', 'region']
NUM = ['marketing_budget', 'planned_discount', 'stock_units', 'list_price',
       'days_in_month', 'lag_1', 'lag_12']

# Històric + juliol (sense vendes) en una sola taula per a calcular lag_1 i lag_12
hist = pd.read_csv('../dades/sales_history.csv')
juliol = pd.read_csv('../dades/sales_forecast_input.csv')
df = pd.concat([hist, juliol], ignore_index=True)
df['month'] = pd.to_datetime(df['month'], format='%Y-%m')
df = df.sort_values(KEY + ['month']).reset_index(drop=True)
for lag in [1, 12]:
    df[f'lag_{lag}'] = df.groupby(KEY)['sales_units'].shift(lag)

def design(frame):  # mateixes variables que el model congelat (CP3, secció 5)
    x = frame[NUM].copy()
    m = frame['month'].dt.month
    x['month_sin'], x['month_cos'] = np.sin(2*np.pi*m/12), np.cos(2*np.pi*m/12)
    x['time_index'] = (frame['month'].dt.year - 2021) * 12 + m - 1
    for p in ['urban', 'mountain', 'electric']:
        x[f'{p}_sin'] = x['month_sin'] * (frame['product_line'] == p)
        x[f'{p}_cos'] = x['month_cos'] * (frame['product_line'] == p)
    return pd.concat([x, pd.get_dummies(frame[KEY + ['campaign']], dtype=float)], axis=1)

# Model congelat: Ridge alpha=300, entrenat amb 2022-2024
train = df[(df['month'] < '2025-01-01')].dropna(subset=['lag_1', 'lag_12'])
x = design(train)
med = x.median(); x = x.fillna(med)
mu, sd = x.mean(), x.std().replace(0, 1)
z = np.column_stack([np.ones(len(x)), ((x - mu) / sd).to_numpy(float)])
pen = np.eye(z.shape[1]) * 300; pen[0, 0] = 0
beta = np.linalg.solve(z.T @ z + pen, z.T @ train['sales_units'].to_numpy())

# Predicció de juliol 2026 (limitada entre 0 i l'estoc inicial)
jul = df[df['month'] == '2026-07-01'].copy()
xj = design(jul).reindex(columns=x.columns, fill_value=0).fillna(med)
zj = np.column_stack([np.ones(len(xj)), ((xj - mu) / sd).to_numpy(float)])
jul['prediccio_unitats'] = np.clip(zj @ beta, 0, jul['stock_units']).round().astype(int)
jul['venda_real'] = np.nan   # encara no la sabem: s'omplirà a final de juliol
jul['tipus'] = 'previsió'

prediccions = jul[['month'] + KEY + ['prediccio_unitats', 'venda_real', 'tipus']].copy()
prediccions['month'] = prediccions['month'].dt.strftime('%Y-%m')
prediccions.to_csv('../evidencies/prediccions_juliol_2026.csv', index=False)
print(len(prediccions), 'prediccions guardades')
prediccions

18 prediccions guardades


,month,channel,product_line,region,prediccio_unitats,venda_real,tipus
66,2026-07,ecommerce,electric,coast,106,NaN,previsió
133,2026-07,ecommerce,electric,inland,76,NaN,previsió
200,2026-07,ecommerce,electric,metro,105,NaN,previsió
267,2026-07,ecommerce,mountain,coast,157,NaN,previsió
334,2026-07,ecommerce,mountain,inland,110,NaN,previsió
401,2026-07,ecommerce,mountain,metro,188,NaN,previsió
468,2026-07,ecommerce,urban,coast,214,NaN,previsió
535,2026-07,ecommerce,urban,inland,154,NaN,previsió
602,2026-07,ecommerce,urban,metro,190,NaN,previsió
669,2026-07,store,electric,coast,92,NaN,previsió


### <span style="color:#D68910">5.2 Recomanació d'estoc</span>

**Reforçar l'estoc (≈ +15 %) en les 6 combinacions on la previsió arriba al 94–100 % de l'estoc**, sobretot en botiga (urban inland, mountain metro, electric inland, mountain coast). En canvi, ecommerce urban coast i ecommerce electric metro en tenen de sobra (s'usa ≈ 60 %) i poden cedir-ne.

- **Per què:** juliol és el mes amb més rotures d'estoc (≈ 37 %), i en 4 combinacions la previsió ja toca el límit, així que la demanda real pot ser més alta.
- **Confiança: mitjana.** L'error és d'≈ 12 u. per combinació en validació i puja a ≈ 16 u. en la prova final (gener–juny 2026), i el model **es queda curt** en mesos forts. Quan es ven tot l'estoc no sabem la demanda real.

### <span style="color:#D68910">5.3 Recomanació de campanya</span>

**Mantindre la campanya *summer* i estendre-la a les 2 combinacions que no en tenen** (ecommerce urban metro i ecommerce electric metro), que a més tenen estoc de sobra. **No pujar** el pressupost de màrqueting ni el descompte.

- **Per què:** a igualtat de la resta, *summer* s'associa a ≈ +11 u. per combinació. En canvi, +1.000 € de màrqueting només en dona ≈ +3 i +5 punts de descompte ≈ +3, i tots dos costen diners.
- **Confiança: baixa.** Les dades són sintètiques: el model aprèn relacions, **no demostra causes**. A més, *summer* només apareix a l'estiu, així que una part de l'efecte pot ser de la temporada i no de la campanya.

In [2]:
# Evidència 1: combinacions on la previsió de juliol s'acosta a l'estoc inicial
jul['us_estoc_%'] = (100 * jul['prediccio_unitats'] / jul['stock_units']).round().astype(int)
print(jul.sort_values('us_estoc_%', ascending=False)[KEY + ['stock_units', 'prediccio_unitats', 'us_estoc_%']].head(6).to_string(index=False))

juliols = df[(df['month'].dt.month == 7) & df['sales_units'].notna()]
print()
print("Juliols anteriors amb rotura d'estoc:", round(100 * juliols['stockout'].mean()), '%')

# Evidència 2: unitats extra que el model associa a cada palanca
efecte = pd.Series(beta[1:] / sd.values, index=x.columns)
print()
print('+1.000 € de màrqueting  →', round(efecte['marketing_budget'] * 1000, 1), 'u.')
print('+5 punts de descompte   →', round(efecte['planned_discount'] * 0.05, 1), 'u.')
print('Campanya summer vs cap  →', round(efecte['campaign_summer'] - efecte['campaign_none'], 1), 'u.')

  channel product_line region  stock_units  prediccio_unitats  us_estoc_%
    store        urban inland          112                112         100
    store     mountain  metro          108                108         100
    store     electric inland           45                 45         100
    store     mountain  coast          136                134          99
ecommerce        urban inland          159                154          97
ecommerce     mountain inland          117                110          94

Juliols anteriors amb rotura d'estoc: 37 %

+1.000 € de màrqueting  → 2.7 u.
+5 punts de descompte   → 2.7 u.
Campanya summer vs cap  → 11.4 u.
